In [51]:
import pandas as pd
import matplotlib.pyplot as plt


In [52]:
df = pd.read_csv(r'C:\Users\123\OneDrive\Desktop\quest_proj\Womens Clothing E-Commerce Reviews.csv')
print("shape: ",df.shape)
print("columns: ",df.columns.tolist())
print("is null: ",df.isna().sum())

shape:  (23486, 11)
columns:  ['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
is null:  Unnamed: 0                    0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64


In [53]:
print(df["Rating"].value_counts().sort_index())

Rating
1      842
2     1565
3     2871
4     5077
5    13131
Name: count, dtype: int64


In [54]:
print(df["Recommended IND"].value_counts(normalize=True))

Recommended IND
1    0.822362
0    0.177638
Name: proportion, dtype: float64


In [55]:
print(df.groupby("Department Name")["Recommended IND"].mean())

Department Name
Bottoms     0.851277
Dresses     0.808197
Intimate    0.850144
Jackets     0.836240
Tops        0.815151
Trend       0.739496
Name: Recommended IND, dtype: float64


In [56]:
print(df[["Title", "Review Text","Rating"]].head(5))

                     Title                                        Review Text  \
0                      NaN  Absolutely wonderful - silky and sexy and comf...   
1                      NaN  Love this dress!  it's sooo pretty.  i happene...   
2  Some major design flaws  I had such high hopes for this dress and reall...   
3         My favorite buy!  I love, love, love this jumpsuit. it's fun, fl...   
4         Flattering shirt  This shirt is very flattering to all due to th...   

   Rating  
0       4  
1       5  
2       3  
3       5  
4       5  


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

In [58]:
df = df.drop(columns=["Unnamed: 0"])


In [59]:
df["Title"]= df["Title"].fillna("")
df["Review Text"]= df["Review Text"].fillna("")
df["text"] = (df["Title"]+" "+df["Review Text"]).str.strip()
df = df[df["text"]!= ""]

In [60]:
X_train , X_test, y_train, y_test = train_test_split(df["text"], df["Recommended IND"], test_size=0.2, random_state=42, stratify=df["Recommended IND"])
vec = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1,2))
Xtr = vec.fit_transform(X_train)
Xte= vec.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(Xtr, y_train)
print(classification_report(y_test,model.predict(Xte)))

words = np.array(vec.get_feature_names_out())
order = np.argsort(model.coef_[0])
print("Top negetive words: ",words[order[:20]])
print("Top positive words: ",words[order[-20:]])


              precision    recall  f1-score   support

           0       0.64      0.84      0.73       820
           1       0.96      0.90      0.93      3709

    accuracy                           0.89      4529
   macro avg       0.80      0.87      0.83      4529
weighted avg       0.90      0.89      0.89      4529

Top negetive words:  ['disappointed' 'wanted love' 'returned' 'huge' 'returning' 'unflattering'
 'cheap' 'poor' 'unfortunately' 'return' 'bad' 'wanted' 'disappointing'
 'fabric' 'meh' 'way' 'going' 'strange' 'material' 'excited']
Top positive words:  ['bit' 'happy' 'easy' 'perfectly' 'lovely' 'comfy' 'fun' 'amazing' 'nice'
 'gorgeous' 'beautiful' 'fits' 'compliments' 'soft' 'little' 'unique'
 'comfortable' 'great' 'perfect' 'love']


In [61]:
neg = df[df["Recommended IND"] == 0]
print(neg["Department Name"].value_counts())
print(neg[neg["Department Name"] == "Trend"]["text"].head(5).tolist())

Department Name
Tops        1902
Dresses     1197
Bottoms      551
Intimate     254
Jackets      167
Trend         30
Name: count, dtype: int64
["This skirt is not what i was expecting at all. for starters it runs large, i expected it to sit at the natural waist and it is definitely made to do so, but it was a good two inches too big and therefore wouldn't lay flat and gaped at the front. no bueno. my biggest gripe, though, is that the color looks nothing like it does in the picture. they have it paired with an ivory top and the two look like they match so well but do not be fooled, the background color of this skirt is not even close to", 'Odd cut I order from retailer regularly and am pretty consistently a 0 petite or xxs petite. i ordered this in a 0 petite and it was like 5 inches too small in the rib cage and chest area. even if i would have got it zipped up, the halter cut is cut narrow so you can\'t wear a bra for sure...strapless would be necessary. also, the actual neck part w

In [62]:
def build_prompt(dept, n=40):
    subset = neg[neg["Department Name"] == dept]
    reviews = subset["text"].sample(min(n, len(subset)), random_state=42).tolist()
    reviews = [r[:300] for r in reviews]   # har review ke pehle 300 characters
    joined = "\n".join(f"- {r}" for r in reviews)
    return f"""You are helping a fashion store owner. Below are customer reviews from the "{dept}" department where the customer did NOT recommend the product.

Reviews:
{joined}

Task:
1. List the top 3 complaint themes and roughly how many reviews mention each.
2. Give one short example phrase for each theme.
3. Suggest one practical fix per theme.
Be concise and use only what is in the reviews."""

prompt = build_prompt("Trend", n=30)
print(prompt)

You are helping a fashion store owner. Below are customer reviews from the "Trend" department where the customer did NOT recommend the product.

Reviews:
- Lovely but the cut/sizing is not good This is a beautiful spring/summer dress, and the quality of the materials is great. i wanted to love it. but the sizing is ridiculous! i'm 5'3" and 115 and i think you could fit three of me comfortably in the xs. while the pictures of the model in the dress don'
- It's like a work of art, but sadly, not flattering It's like a work of art, but sadly, not flattering on me. i absolutely loved the look, and even considered taking it to a tailor, but the material overall is very stiff. if you're teeny tiny and need something to give you the illusion of hips, you m
- Disappointed I was so in love with these when i saw them online. they arrived today, and while the fit was great, i was disappointed. the colors are definitely more drab in person than what's pictured in the photo. the material is very th

In [63]:
trend = neg[neg["Department Name"] == "Trend"]["text"].str.lower()

themes = {
    "size/fit": ["size", "sizing", "fit", "runs", "too big", "too small", "tight", "large", "small", "huge", "massive"],
    "not as pictured": ["pictured", "picture", "photo", "online", "pics", "in person"],
    "fabric/quality": ["fabric", "material", "cheap", "quality", "stiff", "itchy", "thick", "heavy", "see through", "see-through"],
}

for name, words in themes.items():
    count = trend.apply(lambda t: any(w in t for w in words)).sum()
    print(name, count, "of", len(trend))

size/fit 21 of 30
not as pictured 17 of 30
fabric/quality 18 of 30


In [64]:
themes = {
    "size/fit": ["size", "sizing", "fit", "runs", "too big", "too small", "tight", "large", "small", "huge", "massive"],
    "not as pictured": ["pictured", "picture", "photo", "online", "pics", "in person"],
    "fabric/quality": ["fabric", "material", "cheap", "quality", "stiff", "itchy", "thick", "heavy", "see through", "see-through"],
}

def theme_counts(dept):
    texts = neg[neg["Department Name"] == dept]["text"].str.lower()
    result = {name: round(texts.apply(lambda t: any(w in t for w in words)).mean() * 100)
              for name, words in themes.items()}
    return len(texts), result

for dept in neg["Department Name"].dropna().unique():
    n, res = theme_counts(dept)
    print(dept, "| reviews:", n, "|", res)

Dresses | reviews: 1197 | {'size/fit': 67, 'not as pictured': 25, 'fabric/quality': 57}
Intimate | reviews: 254 | {'size/fit': 62, 'not as pictured': 15, 'fabric/quality': 50}
Tops | reviews: 1902 | {'size/fit': 61, 'not as pictured': 26, 'fabric/quality': 51}
Bottoms | reviews: 551 | {'size/fit': 72, 'not as pictured': 20, 'fabric/quality': 54}
Jackets | reviews: 167 | {'size/fit': 64, 'not as pictured': 26, 'fabric/quality': 54}
Trend | reviews: 30 | {'size/fit': 70, 'not as pictured': 57, 'fabric/quality': 60}
